# Hypothesis check

The motivation for this hypothesis is profoundly based on the properties of the AUC and calibrated PD-models.

On one hand the AUC measures the discrimination power of a scoring system. This has nothing to do with the scoring system being calibrated.

However a model that calculates well the probability of default, meaning a well calibrated model $p$ in the sense
$$
p(x) \approx \mathbb{P}(Y=1 | X = x),
$$

should be a suitable basis to predict "an expected AUC" through monte carlo simulations. If this is true at least
the unbiased logit estimator for the very simple GDP should usually give this results.

## Checking out the default run

In [1]:
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditData, CreditDataGenerator
from berebasl.estimation.bayesian_evaluation import batched_auroc
from berebasl.estimation.classifiers import TorchLogistic

In [2]:
device = torch.device(
        "cuda" if torch.cuda.is_available() else 
        "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else 
        "cpu"
    )

sim_path = '../berebasl/data/simulations/default_sim'

sim_results_path = os.path.join(sim_path, 'simulation_results.pt')
init_objs_path = os.path.join(sim_path, 'initial_simulation_objects.pt')
sim_results = torch.load(sim_results_path, map_location='cpu', weights_only=False)
init_objs = torch.load(init_objs_path, map_location='cpu', weights_only=False)

credit_data: CreditData = sim_results["credit_data"].to(device)
data_generator: CreditDataGenerator = init_objs['data_generator']

In [5]:
_, lbls = credit_data.accepts()
print("Checking out all accepts")
accept_count = lbls.size(0)
print("\tCount accepts:", accept_count)
defaults = lbls.sum().to(int).item()
print("\tDefaults among accepts:", defaults)
print("\tRate of Defaults among accepts:", defaults/accept_count)

_, lbls = credit_data.accepts(from_round_idx=1)
print("Checking out accepts without the first round")
accept_count = lbls.size(0)
print("\tCount accepts:", accept_count)
defaults = lbls.sum().to(int).item()
print("\tDefaults among accepts:", defaults)
print("\tRate of Defaults among accepts:", defaults/accept_count)


Checking out all accepts
	Count accepts: 6020
	Defaults among accepts: 144
	Rate of Defaults among accepts: 0.023920265780730896
Checking out accepts without the first round
	Count accepts: 5980
	Defaults among accepts: 140
	Rate of Defaults among accepts: 0.023411371237458192


In [3]:
#data_generator: CreditDataGenerator = init_objs['data_generator']

data_generator.bayes_error_rate()

0.1298000067472458

In [3]:
last_gen_round_idx = credit_data.last_gen_round.item()
for state_dict_gen in sim_results['classifiers_state_dicts']:
    gen_round_idx = state_dict_gen["gen_round_nr"] - 1
    if gen_round_idx == last_gen_round_idx:
        continue
    oracle_classifier = TorchLogistic.instantiate_from_state_dict(state_dict_gen['classifiers']['oracle_classifier'], device)
    feats_acc, labels_acc = credit_data.accepts(include_gen_round=False, up_to_round_idx=gen_round_idx)
    feats_acc_next_round, labels_acc_next_round = credit_data.accepts(include_gen_round=False, from_round_idx=gen_round_idx+1, up_to_round_idx=gen_round_idx+1)
    break

In [27]:
_, lbls_accept_after_init = credit_data.accepts(from_round_idx=1)

lbls_accept_after_init.sum()/lbls_accept_after_init.size(0)

tensor(0.9916, dtype=torch.float64)

In [7]:
oracle_classifier.predict_proba(feats_acc_next_round)[:,1]

tensor([0.9513, 0.9756, 0.8750, 0.8948, 0.9997, 0.9122, 0.9984, 0.9970, 0.9993,
        0.9888, 0.9987, 0.9401, 0.9999, 0.9968, 0.9817, 0.9701, 0.9523, 0.9911,
        0.9916, 1.0000], dtype=torch.float64, grad_fn=<SelectBackward0>)

In [17]:
state_dict.keys()

dict_keys(['weight', 'bias'])

In [ ]:
if True:
    feats_acc, labels_acc = credit_data.accepts(include_gen_round=False, up_to_round_idx=gen_round_idx)

    feats_acc_next_round, labels_acc_next_round = credit_data.accepts(include_gen_round=False, from_round_idx=gen_round_idx+1, up_to_round_idx=gen_round_idx+1)

## Step 1: Make GDP less easy

The GDP shows a too well linearly separable problem, such that it is highly unrealistic. A first generation showed highly unrealistic false positive rate of **98,57%** among accepts, meaning the decision was right almost always, among those a lot of the defaults come from the initial generation round, such that when leaving it out the FPR was **99,16%** -> absolute nonsense

In [7]:
k = 2
mixture = dgp.good_mixture
k = mixture.mean.size(-1)
mixture.cov_chol_decomp.expand(mixture.b, mixture.m, k, k)

tensor([[[[ 1.0000,  0.0000],
          [-0.2000,  0.9798]]]])

In [2]:
from berebasl.simulation.credit_data_simulation import CreditDataGenerator
from berebasl.simulation.acceptance_loop import default_dgp

dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else 
    "cpu"
)

dgp = default_dgp(seed_credit_data_gen=1807, deterministic_weights_for_mixture_sampling=True, device=device, dtype=dtype)

In [4]:
print("mu_good =", dgp.good_mixture.mean, "mu_bad =", dgp.bad_mixture.mean)
print("sigma_good = ", dgp.good_mixture.cov)
print("sigma_bad =", dgp.bad_mixture.cov)

mu_good = tensor([1., 2.]) mu_bad = tensor([0., 0.])
sigma_good =  tensor([[ 1.0000, -0.2000],
        [-0.2000,  1.0000]])
sigma_bad = tensor([[1.0000, 0.2000],
        [0.2000, 1.0000]])


In [ ]:
device = torch.device(
        "cuda" if torch.cuda.is_available() else 
        "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else 
        "cpu"
    )

sim_path = '../berebasl/data/simulations/simulation_20260215_085247'

sim_results_path = os.path.join(sim_path, 'simulation_results.pt')
init_objs_path = os.path.join(sim_path, 'initial_simulation_objects.pt')
sim_results = torch.load(sim_results_path, map_location='cpu', weights_only=False)
init_objs = torch.load(init_objs_path, map_location='cpu', weights_only=False)

credit_data: CreditData = sim_results["credit_data"].to(device)
data_generator: CreditDataGenerator = init_objs['data_generator']

torch.Size([4, 1, 3, 3])

In [ ]:
last_classifiers = sim_results['classifiers_state_dicts'][-1]
accepts_classifier = TorchLogistic.instantiate_from_state_dict(last_classifiers["classifiers"]["accepts_classifier"])
oracle_classifier = TorchLogistic.instantiate_from_state_dict(last_classifiers["classifiers"]["oracle_classifier"])

In [3]:
accepts_based_classif = TorchLogistic(
    n_features=credit_data.features_count, 
    dtype=credit_data.features.dtype,
    device=credit_data.device
)
oracle_classif = TorchLogistic(
    n_features=credit_data.features_count, 
    dtype=credit_data.features.dtype,
    device=credit_data.device
)

mc_n = 1000
total_gens = credit_data.last_gen_round.item() + 1
gen_count_ahead_of_time = round(total_gens*0.1)
min_gens_passed = round(total_gens*0.2)

min_bads = 4

diagnostics = []


for hist_up_to_gen_idx in range(min_gens_passed, total_gens-gen_count_ahead_of_time):
    available_hist_data = credit_data.to_sample_dataset(up_to_round_idx=hist_up_to_gen_idx)
    accepts_based_classif.fit(available_hist_data.features_labeled, available_hist_data.labels)

    feats_unbiased, lbls_unbiased = credit_data.unbiased_obs(up_to_round_idx=hist_up_to_gen_idx)
    oracle_classif.fit(feats_unbiased, lbls_unbiased)

    future_gens_data = credit_data.to_sample_dataset(
        from_round_idx=hist_up_to_gen_idx+1, 
        up_to_round_idx=hist_up_to_gen_idx+gen_count_ahead_of_time
    )
    bads_among_accepts = int(future_gens_data.labels.sum())
    if bads_among_accepts < min_bads:
        continue

    future_gens_data.manual_seed(1807 + hist_up_to_gen_idx)


    biased_posterior_future_acc = accepts_based_classif.predict_proba(future_gens_data.features_labeled)[..., 1]
    unbiased_posterior_future_acc = oracle_classif.predict_proba(future_gens_data.features_labeled)[..., 1]

    expanded_biased_post = biased_posterior_future_acc.unsqueeze(0).expand(mc_n, -1)
    pseudo_labels_biased = torch.bernoulli(expanded_biased_post,
                                           generator=future_gens_data.rng)
    pseudo_labels_unbiased = torch.bernoulli(unbiased_posterior_future_acc.unsqueeze(0).expand(mc_n, -1),
                                             generator=future_gens_data.rng)
    
    auroc_biased_pseudo_lbls = batched_auroc(preds=expanded_biased_post, targets=pseudo_labels_biased).mean().item()
    auroc_unbiased_pseudo_lbls = batched_auroc(preds=expanded_biased_post, targets=pseudo_labels_unbiased).mean().item()

    
    
    bootstrap_fut_lbls_idx = torch.randint(future_gens_data.count_labeled, size=(mc_n, future_gens_data.count_labeled),
                                    generator=future_gens_data.rng)
    bootstrap_fut_lbls = future_gens_data.labels[bootstrap_fut_lbls_idx]
    mask_less_than_min_bads = bootstrap_fut_lbls.sum(dim=-1) < min_bads

    while mask_less_than_min_bads.any():
        bootstrap_fut_lbls_idx[mask_less_than_min_bads] = torch.randint(
            future_gens_data.count_labeled, 
            size=(mask_less_than_min_bads.sum().item(), future_gens_data.count_labeled),
            generator=future_gens_data.rng
        )
        bootstrap_fut_lbls = future_gens_data.labels[bootstrap_fut_lbls_idx]
        mask_less_than_min_bads = bootstrap_fut_lbls.sum(dim=-1) < min_bads

    bootstrap_fut_feats = future_gens_data.features_labeled[bootstrap_fut_lbls_idx] # [mc_n, N_f, F]
    bootstrap_post_biased = accepts_based_classif.predict_proba(bootstrap_fut_feats)[..., 1] # [mc_n, N_f]
    bootstrap_post_unbiased = oracle_classif.predict_proba(bootstrap_fut_feats)[..., 1] # [mc_n, N_f]

    auroc_biased_bs_lbls = torch.stack(
        [batched_auroc(preds=bootstrap_post_biased[b], targets=bootstrap_fut_lbls[b]) for b in range(mc_n)]
    ).mean().item()
    auroc_unbiased_bs_lbls = torch.stack(
        [batched_auroc(preds=bootstrap_post_unbiased[b], targets=bootstrap_fut_lbls[b]) for b in range(mc_n)]
    ).mean().item()

    auroc_biased_obs_lbls = batched_auroc(biased_posterior_future_acc, future_gens_data.labels).item()
    auroc_unbiased_obs_lbls = batched_auroc(unbiased_posterior_future_acc, future_gens_data.labels).item()

    
    fut_feats_unbiased, fut_lbls_unbiased = credit_data.unbiased_obs(
        from_round_idx=hist_up_to_gen_idx+1, 
        up_to_round_idx=hist_up_to_gen_idx+gen_count_ahead_of_time
    )

    auroc_full_new_gen_biased = batched_auroc(
        accepts_based_classif.predict_proba(fut_feats_unbiased)[..., 1],
        fut_lbls_unbiased
    ).item()
    auroc_full_new_gen_unbiased = batched_auroc(
        oracle_classif.predict_proba(fut_feats_unbiased)[..., 1],
        fut_lbls_unbiased
    ).item()

    diagnostics.append({
        "unbiased_auroc_pseudo" : auroc_unbiased_pseudo_lbls, "biased_auroc_pseudo" : auroc_biased_pseudo_lbls,
        "unbiased_auroc_observed" : auroc_unbiased_obs_lbls, "biased_auroc_observed" : auroc_biased_obs_lbls,
        "unbiased_auroc_boot" : auroc_unbiased_bs_lbls, "biased_auroc_boot" : auroc_biased_bs_lbls,
        "unbiased_auroc_full" : auroc_full_new_gen_unbiased, "biased_auroc_full" : auroc_full_new_gen_biased
    })

    if (hist_up_to_gen_idx-min_gens_passed+1)% 10 == 0:
        print("Iterations to go:", total_gens-gen_count_ahead_of_time - hist_up_to_gen_idx + 1)

    break

In [3]:
accepts_based_classif = TorchLogistic(
    n_features=credit_data.features_count, 
    dtype=credit_data.features.dtype,
    device=credit_data.device
)
oracle_classif = TorchLogistic(
    n_features=credit_data.features_count, 
    dtype=credit_data.features.dtype,
    device=credit_data.device
)

mc_n = 1000
total_gens = credit_data.last_gen_round.item() + 1
gen_count_ahead_of_time = round(total_gens*0.1)
min_gens_passed = round(total_gens*0.2)

min_bads = 4

diagnostics = []


for hist_up_to_gen_idx in range(min_gens_passed, total_gens-gen_count_ahead_of_time):
    available_hist_data = credit_data.to_sample_dataset(up_to_round_idx=hist_up_to_gen_idx)
    break

In [27]:
N = 27
k = 4

N_perm = torch.arange(N)+1#torch.randperm(N, generator=rng, device=features.device) + 1
    
N_mod_k = N % k

torch.cat([
        N_perm[:-N_mod_k].reshape(k, N//k),
        torch.nn.functional.pad(N_perm[-N_mod_k:], pad=(k-N_mod_k, 0), value=0).unsqueeze(-1)
    ], dim=-1)

tensor([[ 1,  2,  3,  4,  5,  6,  0],
        [ 7,  8,  9, 10, 11, 12, 25],
        [13, 14, 15, 16, 17, 18, 26],
        [19, 20, 21, 22, 23, 24, 27]])

In [ ]:
from typing import Union
def k_fold_cv_normalized_split(features: torch.Tensor, labels: torch.Tensor, 
                               rng: torch.Generator, k: int = 4, min_bad: int = 4,
                               nan_lbls : Union[float, int] = float('nan'),
                               safety_checks: bool = True):
    if safety_checks and (not 
            (features.dim()-1 == labels.dim() == 1) and 
            (features.size(0)==labels.size(0)) and
            (features.device==labels.device)):
        raise AssertionError("Features expected to be 2d and labels 1d and have same leadin dimension")
    
    N = labels.size(0)
    N_perm = torch.randperm(N, generator=rng, device=features.device) + 1
    
    nan_padded_feats = torch.nn.functional.pad(features, pad=(0,0,1,0))
    nan_padded_labels = torch.nn.functional.pad(labels, pad=(1,0), value=nan_lbls)

    
    N_mod_k = N % k
    cv_idx = N_perm.reshape(k, N//k) if N_mod_k == 0 else torch.cat([
        N_perm[:-N_mod_k].reshape(k, N//k),
        torch.nn.functional.pad(N_perm[-N_mod_k:], pad=(k-N_mod_k, 0), value=0).unsqueeze(-1)
    ], dim=-1)

    cv_feats = nan_padded_feats[cv_idx]
    cv_lbls = nan_padded_labels[cv_idx]

    return cv_feats, cv_lbls

cv_feats, cv_lbls = k_fold_cv_normalized_split(available_hist_data.features_labeled, available_hist_data.labels, rng=available_hist_data.rng)

(torch.Size([4, 310, 2]), torch.Size([4, 310]))

In [ ]:
def batched_auroc_old(
    preds: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Batched AUROC computed via explicit ROC construction with
    grouped thresholds and trapezoidal integration.

    AUROC is computed independently along the last dimension,
    treating all leading dimensions as batch dimensions
    (analoguous to nn.Linear-style semantics).

    Matches torchmetrics.BinaryAUROC semantics.

    Args:
        preds:   Tensor of shape (*batch_dims, N), prediction scores
        targets: Tensor of shape (*batch_dims, N), binary labels {0,1}

    Returns:
        auc: Tensor of shape (*batch_dims), AUROC per mini-dataset
    """
    if preds.shape != targets.shape:
        raise ValueError("preds and targets must have the same shape")

    *batch_dims, N = preds.shape
    device = preds.device

    # Flatten batch dimensions
    B = int(torch.tensor(batch_dims).prod()) if batch_dims else 1
    preds = preds.reshape(B, N)
    targets = targets.reshape(B, N)

    # Sort by descending score
    order = preds.argsort(dim=-1, descending=True)
    sorted_preds = preds.gather(dim=-1, index=order)
    sorted_targets = targets.gather(dim=-1, index=order)


    # Count positives / negatives
    P = sorted_targets.sum(dim=-1)           # [B]
    Q = N - P                                # [B]

    # Cumulative true / false positives
    tps = torch.cumsum(sorted_targets, dim=-1)
    fps = torch.cumsum(1 - sorted_targets, dim=-1)

    # Identify score changes (grouped thresholds)
    score_change = torch.ones_like(sorted_preds, dtype=torch.bool)
    score_change[:, 1:] = sorted_preds[:, 1:] != sorted_preds[:, :-1]

    # Select ROC vertices
    tps = tps[score_change].view(B, -1)
    fps = fps[score_change].view(B, -1)

    # Normalize to TPR / FPR
    tpr = tps / P.unsqueeze(-1)
    fpr = fps / Q.unsqueeze(-1)

    # Explicit (0,0) start point
    zero = torch.zeros(B, 1, device=device)
    tpr = torch.cat([zero, tpr], dim=-1)
    fpr = torch.cat([zero, fpr], dim=-1)

    # Trapezoidal integration
    #return tpr, fpr
    auc = torch.trapz(tpr, fpr, dim=-1)

    # Reshape back to batch dimensions
    auc = auc.reshape(*batch_dims) if batch_dims else auc[0] # get 0 dim tensor in this case

    return auc

auc = batched_auroc(preds=expanded_biased_post, targets=pseudo_labels_biased)

In [ ]:



auc_orig = [batched_auroc(bootstrap_post_biased[b], bootstrap_fut_lbls[b]) for b in range(bootstrap_post_biased.size(0))]
auc_orig

tensor(0.6783, dtype=torch.float64)

In [226]:
(bootstrap_post_biased*1e5).to(int).isnan()

tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]])

In [233]:
from enum import Enum

class Validations(Enum):
    same_dtype = "same_dtype"

Validations("same_dtype").value

'same_dtype'

In [146]:
a_tensor = torch.randn((B, N))
zeros = torch.zeros((B,1), dtype=torch.float64)
torch.cat([zeros, a_tensor], dim=1)

tensor([[ 0.0000, -0.3354,  0.5121,  ...,  1.8869, -0.3513,  1.5643],
        [ 0.0000, -0.6133,  0.5739,  ..., -2.2968, -0.2899, -0.6186],
        [ 0.0000, -1.0512, -0.1812,  ..., -1.9413,  1.3514, -1.2860],
        ...,
        [ 0.0000,  0.0084, -0.8167,  ..., -0.5943, -0.6031,  1.2424],
        [ 0.0000,  0.7373,  0.9913,  ...,  0.0280,  0.5700, -0.5885],
        [ 0.0000, -1.7444,  1.1904,  ...,  0.9945,  0.0311,  0.0993]],
       dtype=torch.float64)

In [142]:
torch.nn.functional.pad(last_valid_per_batch[:-1], pad=(1,0), mode='constant', value=0)

tensor([  0, 390, 376, 382, 373, 375, 382, 373, 373, 389, 372, 367, 381, 378,
        381, 375, 380, 378, 379, 384, 366, 365, 376, 371, 395, 382, 381, 374,
        362, 369, 382, 390, 390, 360, 382, 382, 381, 370, 384, 380, 379, 382,
        379, 384, 375, 372, 399, 387, 380, 386, 378, 383, 382, 379, 373, 384,
        370, 378, 375, 372, 383, 377, 370, 376, 381, 384, 369, 386, 368, 380,
        373, 380, 373, 386, 378, 379, 386, 380, 363, 386, 374, 383, 386, 390,
        392, 385, 385, 381, 379, 387, 376, 372, 386, 373, 378, 379, 385, 387,
        380, 374, 380, 369, 368, 376, 381, 375, 373, 374, 383, 387, 380, 375,
        372, 374, 377, 374, 388, 374, 376, 382, 381, 367, 376, 376, 371, 391,
        379, 373, 372, 372, 384, 372, 385, 387, 381, 398, 386, 368, 382, 388,
        384, 381, 382, 364, 374, 374, 381, 381, 386, 372, 373, 385, 368, 378,
        381, 381, 386, 378, 385, 392, 381, 386, 387, 377, 381, 383, 373, 376,
        373, 373, 377, 386, 380, 376, 375, 383, 372, 394, 387, 3

In [132]:
tpr = tps[b][score_change[b]] / P[b].unsqueeze(-1)
fpr = fps[b][score_change[b]] / Q[b].unsqueeze(-1)

# Explicit (0,0) start point
zero = torch.tensor([0])
tpr = torch.cat([zero, tpr], dim=-1)
fpr = torch.cat([zero, fpr], dim=-1)

# Trapezoidal integration
auc = torch.trapz(tpr, fpr, dim=-1)
auc == auc_orig

tensor(True)

In [90]:
flattened_idx[[767, 768]]

tensor([782, 812])

In [122]:
idx_b_n = all_flattend_idxs_padded_uniques.gather(dim=-1, index=idx_last_valid_per_batch.unsqueeze(-1))
changes_in_flatten = idx_b_n[1:] - torch.cumsum((max_trues-1) - last_valid_per_batch, dim = -1).unsqueeze(-1)[:-1] + torch.tensor([[0,1]])

flattened_idx[changes_in_flatten[:-1]]

tensor([[   782,    812],
        [  1194,   1218],
        [  1591,   1624],
        ...,
        [404763, 404782],
        [405155, 405188],
        [405561, 405594]])

In [ ]:
_, flat = torch.where(flattened_idx.unsqueeze(0) == idx_b_n)
 .squeeze(-1) - flat

tensor([    0,    15,    44,    67,    99,   129,   152,   184,   216,   232,
          265,   303,   327,   354,   378,   408,   433,   460,   486,   507,
          546,   586,   615,   649,   659,   682,   706,   737,   780,   816,
          839,   854,   869,   914,   937,   960,   984,  1019,  1040,  1065,
         1091,  1114,  1140,  1161,  1191,  1224,  1230,  1248,  1273,  1292,
         1319,  1341,  1364,  1390,  1422,  1443,  1478,  1505,  1535,  1568,
         1590,  1618,  1653,  1682,  1706,  1727,  1763,  1782,  1819,  1844,
         1876,  1901,  1933,  1952,  1979,  2005,  2024,  2049,  2091,  2110,
         2141,  2163,  2182,  2197,  2210,  2230,  2250,  2274,  2300,  2318,
         2347,  2380,  2399,  2431,  2458,  2484,  2504,  2522,  2547,  2578,
         2603,  2639,  2676,  2705,  2729,  2759,  2791,  2822,  2844,  2862,
         2887,  2917,  2950,  2981,  3009,  3040,  3057,  3088,  3117,  3140,
         3164,  3202,  3231,  3260,  3294,  3308,  3334,  3366, 

In [87]:
idx_b_n = all_flattend_idxs_padded_uniques.gather(dim=-1, index=idx_last_valid_per_batch.unsqueeze(-1))
idx_b_n

tensor([[   390],
        [   782],
        [  1194],
        [  1591],
        [  1999],
        [  2412],
        [  2809],
        [  3215],
        [  3637],
        [  4026],
        [  4427],
        [  4847],
        [  5250],
        [  5659],
        [  6059],
        [  6470],
        [  6874],
        [  7281],
        [  7692],
        [  8080],
        [  8485],
        [  8902],
        [  9303],
        [  9733],
        [ 10126],
        [ 10531],
        [ 10930],
        [ 11324],
        [ 11737],
        [ 12156],
        [ 12570],
        [ 12976],
        [ 13352],
        [ 13780],
        [ 14186],
        [ 14591],
        [ 14986],
        [ 15406],
        [ 15808],
        [ 16213],
        [ 16622],
        [ 17025],
        [ 17436],
        [ 17833],
        [ 18236],
        [ 18669],
        [ 19063],
        [ 19462],
        [ 19874],
        [ 20272],
        [ 20683],
        [ 21088],
        [ 21491],
        [ 21891],
        [ 22308],
        [ 

In [25]:
tps.size(0)

1000

In [26]:
tps_sums = [tps[b][score_change[b]][1:] + tps[b][score_change[b]][:-1] for b in range(tps.size(0))]

In [27]:
flattened_idx

tensor([     0,      1,      2,  ..., 405966, 405967, 405968])

In [28]:
tps_flat = tps[score_change]

In [ ]:
def batched_auroc(
    preds: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Batched AUROC computed via explicit ROC construction with
    grouped thresholds and trapezoidal integration.

    AUROC is computed independently along the last dimension,
    treating all leading dimensions as batch dimensions
    (analoguous to nn.Linear-style semantics).

    Matches torchmetrics.BinaryAUROC semantics.

    Args:
        preds:   Tensor of shape (*batch_dims, N), prediction scores
        targets: Tensor of shape (*batch_dims, N), binary labels {0,1}

    Returns:
        auc: Tensor of shape (*batch_dims), AUROC per mini-dataset
    """
    if preds.shape != targets.shape:
        raise ValueError("preds and targets must have the same shape")

    *batch_dims, N = preds.shape
    device = preds.device

    # Flatten batch dimensions
    B = int(torch.tensor(batch_dims).prod()) if batch_dims else 1
    preds = preds.reshape(B, N)
    targets = targets.reshape(B, N)

    # Sort by descending score
    order = preds.argsort(dim=-1, descending=True)
    sorted_preds = preds.gather(dim=-1, index=order)
    sorted_targets = targets.gather(dim=-1, index=order)


    # Count positives / negatives
    P = sorted_targets.sum(dim=-1)           # [B]
    Q = N - P                                # [B]

    # Cumulative true / false positives
    tps = torch.cumsum(sorted_targets, dim=-1)
    fps = torch.cumsum(1 - sorted_targets, dim=-1)

    # Identify score changes (grouped thresholds)
    score_change = torch.ones_like(sorted_preds, dtype=torch.bool)
    score_change[:, 1:] = sorted_preds[:, 1:] != sorted_preds[:, :-1]

    # Select ROC vertices
    tps = tps[score_change].view(B, -1)
    fps = fps[score_change].view(B, -1)

    # Normalize to TPR / FPR
    tpr = tps / P.unsqueeze(-1)
    fpr = fps / Q.unsqueeze(-1)

    # Explicit (0,0) start point
    zero = torch.zeros(B, 1, device=device)
    tpr = torch.cat([zero, tpr], dim=-1)
    fpr = torch.cat([zero, fpr], dim=-1)

    # Trapezoidal integration
    auc = torch.trapz(tpr, fpr, dim=-1)

    # Reshape back to batch dimensions
    auc = auc.reshape(*batch_dims) if batch_dims else auc[0] # get 0 dim tensor in this case

    return auc

def batched_auroc_corrected(
    preds: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Batched AUROC computed via explicit ROC construction with
    grouped thresholds and trapezoidal integration.
    AUROC is computed independently along the last dimension,
    treating all leading dimensions as batch dimensions
    (analoguous to nn.Linear-style semantics).
    Matches torchmetrics.BinaryAUROC semantics.
    Args:
        preds:   Tensor of shape (*batch_dims, N), prediction scores
        targets: Tensor of shape (*batch_dims, N), binary labels {0,1}
    Returns:
        auc: Tensor of shape (*batch_dims), AUROC per mini-dataset
    """
    if preds.shape != targets.shape:
        raise ValueError("preds and targets must have the same shape")
    *batch_dims, N = preds.shape
    device = preds.device

    # Flatten batch dimensions
    B = int(torch.tensor(batch_dims).prod()) if batch_dims else 1
    preds = preds.reshape(B, N)
    targets = targets.reshape(B, N)

    # Sort by descending score
    order = preds.argsort(dim=-1, descending=True)
    sorted_preds = preds.gather(dim=-1, index=order)
    sorted_targets = targets.gather(dim=-1, index=order)

    # Count positives / negatives
    P = sorted_targets.sum(dim=-1)  # [B]
    Q = N - P                       # [B]

    # Cumulative true / false positives
    tps = torch.cumsum(sorted_targets, dim=-1)
    fps = torch.cumsum(1 - sorted_targets, dim=-1)

    # Identify score changes (grouped thresholds)
    score_change = torch.ones_like(sorted_preds, dtype=torch.bool)
    score_change[:, 1:] = sorted_preds[:, 1:] != sorted_preds[:, :-1]

    # Packed column index for each selected element within its row.
    # cumsum gives 1-based rank among selected elements → subtract 1 for 0-based.
    col_idx = torch.cumsum(score_change, dim=-1) - 1  # [B, N]
    max_len = score_change.sum(dim=-1).max().item()

    # Scatter selected tps/fps into [B, max_len] tensors.
    # Positions where score_change is False write zero and do not corrupt
    # real values since tps/fps are monotonically increasing.
    tps = tps.new_zeros(size=(B, max_len)).scatter_(
        1, col_idx.clamp(max=max_len - 1), tps * score_change
    )
    fps = fps.new_zeros(size=(B, max_len)).scatter_(
        1, col_idx.clamp(max=max_len - 1), fps * score_change
    )

    fps.new_zeros(size=(B, max_len))

    # Normalize to TPR / FPR
    tpr = tps / P.unsqueeze(-1)
    fpr = fps / Q.unsqueeze(-1)

    # Explicit (0, 0) start point
    zero = torch.zeros(B, 1, device=device)
    tpr = torch.cat([zero, tpr], dim=-1)
    fpr = torch.cat([zero, fpr], dim=-1)

    # Trapezoidal integration
    auc = torch.trapz(tpr, fpr, dim=-1)

    # Reshape back to batch dimensions
    auc = auc.reshape(*batch_dims) if batch_dims else auc[0]
    return auc